In [1]:
import pandas as pd

In [ ]:
# df=pd.read_csv("pre.csv")

In [3]:
import dagshub
import mlflow

# Connect to your DagsHub repo
dagshub.init(repo_owner="shreychauhan02", repo_name="my-mlflow-project", mlflow=True)


Accessing as shreychauhan02

Initialized MLflow to track repo "shreychauhan02/my-mlflow-project"

Repository shreychauhan02/my-mlflow-project initialized!

In [4]:
mlflow.set_experiment("BOW vs TfIdf")

2026/09/21 16:25:30 INFO mlflow.tracking.fluent: Experiment with name 'BOW vs TfIdf' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/c6e6ba3fdb4140dfac344a93364c3152', creation_time=1789988131458, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789988131458, lifecycle_stage='active', name='BOW vs TfIdf', tags={}, trace_location=None, workspace='default'>

In [5]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt

import seaborn as sns
import os


In [6]:
df=pd.read_csv("../dataset/pre.csv").dropna(subset=['clean_comment'])

In [7]:
df.shape

(36662, 2)

In [9]:
def run_experiment(vectorizer_type,ngram_range,vectorizer_max_features,vectorizer_name):
    if vectorizer_type=="BoW":
        vectorizer=CountVectorizer(ngram_range=ngram_range,max_features=vectorizer_max_features)
    else:
        vectorizer=TfidfVectorizer(ngram_range=ngram_range,max_features=vectorizer_max_features)

    X_text=df['clean_comment']
    y=df['category']

    # Split the raw text FIRST, then fit the vectorizer on train only (no leakage)
    X_train,X_test,y_train,y_test=train_test_split(X_text,y,test_size=0.2,random_state=42,stratify=y)
    X_train=vectorizer.fit_transform(X_train)
    X_test=vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_name}_{ngram_range}")


# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)]  # unigrams, bigrams, trigrams
max_features = 5000

for ngram_range in ngram_ranges:
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")